In [3]:
!pip install  accelerate sentence-transformers datasets==2.19.1 faiss-cpu pandas numpy  rank-bm25  nltk  rouge-score huggingface_hub hf_transfer tqdm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 5.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.5 MB/s eta

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3090


In [5]:
import torch

print("Torch version:", torch.__version__)
print("CUDA (PyTorch build):", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu130
CUDA (PyTorch build): 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 3090


In [ ]:


from huggingface_hub import login

login("")



In [7]:

!pip install bitsandbytes>=0.46.1


In [8]:
!pip install evaluate rouge-score bert-score nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 15.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.4 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 38.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 37.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.7 MB/s eta 0:00:00:00:01


In [9]:

# ============================
# 1. IMPORTS
# ============================
import os, gc
import numpy as np
import pandas as pd
import torch, faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import evaluate
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ============================
# 2. CONFIG
# ============================
DATASET_NAME = "nvidia/TechQA-RAG-Eval"
TOP_K = 3

MODELS = [
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
]

# ============================
# 3. LOAD DATA
# ============================
full_data = load_dataset(DATASET_NAME, split="train", trust_remote_code=True)
dataset   = full_data.select(range(150))   # eval slice

# ============================
# 4. BUILD CORPUS  ← FIXED
# ============================
docs_db = []

for row in full_data:
    for ctx in row.get("contexts", []):          # contexts is a list of dicts
        txt = ctx.get("text", "")
        if isinstance(txt, str) and len(txt.strip()) > 20:
            docs_db.append(txt[:1500])

# De-duplicate
docs_db = list(dict.fromkeys(docs_db))
print("Total valid docs:", len(docs_db))

# ============================
# 5. EMBEDDINGS + FAISS
# ============================
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

emb = embed_model.encode(
    docs_db,
    batch_size=128,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype("float32")

print("Embedding shape:", emb.shape)
assert len(emb.shape) == 2, f"Bad embedding shape: {emb.shape}"

faiss.normalize_L2(emb)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("FAISS index built:", index.ntotal, "vectors")

# ============================
# 6. RETRIEVE
# ============================
def retrieve(query):
    q = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, idx = index.search(q, TOP_K)
    docs = [docs_db[i] for i in idx[0]]
    return docs, scores[0]

# ============================
# 7. GENERATOR
# ============================
def load_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        name,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model.eval()
    return tok, model


def generate(tok, model, query, context):
    prompt = (
        "You are a helpful technical assistant.\n"
        "Answer the question using ONLY the context below.\n"
        "If the answer is not in the context, say: I don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    inp = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
        )

    generated = out[0][inp["input_ids"].shape[1]:]
    return tok.decode(generated, skip_special_tokens=True).strip()

# ============================
# 8. METRICS
# ============================
def normalize(text):
    return text.lower().strip()

def compute_em(pred, gt):
    return int(normalize(gt) in normalize(pred))

def compute_f1(pred, gt):
    pred_tokens = set(normalize(pred).split())
    gt_tokens   = set(normalize(gt).split())
    if not pred_tokens or not gt_tokens:
        return 0.0
    common = pred_tokens & gt_tokens
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens)
    r = len(common) / len(gt_tokens)
    return 2 * p * r / (p + r)

def compute_recall_embedding(retrieved_docs, gt_contexts):
    if not gt_contexts:
        return None
    doc_emb = embed_model.encode(retrieved_docs, convert_to_numpy=True).astype("float32")
    ctx_emb = embed_model.encode(gt_contexts,    convert_to_numpy=True).astype("float32")
    sims = np.matmul(doc_emb, ctx_emb.T)
    return int(np.max(sims) > 0.6)

# ============================
# NEW METRICS
# ============================
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")
smooth = SmoothingFunction().method1

def compute_bleu(pred, gt):
    return sentence_bleu([gt.split()], pred.split(), smoothing_function=smooth)

def compute_rouge(pred, gt):
    scores = rouge.compute(predictions=[pred], references=[gt])
    return scores["rouge1"], scores["rougeL"]

def compute_bertscore(pred, gt):
    res = bertscore.compute(predictions=[pred], references=[gt], lang="en")
    return res["precision"][0], res["recall"][0], res["f1"][0]

# ============================
# 9. NLI  (Hallucination)
# ============================
nli_model = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=0,
    torch_dtype=torch.float16,
)

def compute_hallucination(context, answer):
    result = nli_model([{"text": context[:512], "text_pair": answer}])[0]
    return 1 - result["score"] if result["label"].upper() == "ENTAILMENT" else 1.0

# ============================
# 10. MAIN LOOP
# ============================
results = []

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print('='*60)

    tok, model = load_model(model_name)

    for i, sample in enumerate(dataset):
        query       = sample.get("question", "")
        gt_answer   = sample.get("answer",   "")
        # FIX: contexts is a list of dicts with a "text" key
        gt_contexts = [c["text"] for c in sample.get("contexts", []) if c.get("text")]

        if not query.strip():
            continue

        # Skip unanswerable questions
        if sample.get("is_impossible", False):
            continue

        docs, scores = retrieve(query)
        context = "\n\n".join(docs)
        pred = generate(tok, model, query, context)

        em  = compute_em(pred, gt_answer)
        f1  = compute_f1(pred, gt_answer)
        recall = compute_recall_embedding(docs, gt_contexts)
        hallucination = compute_hallucination(context, pred)

        bleu = compute_bleu(pred, gt_answer)
        rouge1, rougeL = compute_rouge(pred, gt_answer)
        bert_p, bert_r, bert_f1 = compute_bertscore(pred, gt_answer)

        print(f"[{i}] EM:{em} | F1:{f1:.2f} | BERT:{bert_f1:.2f} | Hal:{hallucination:.3f} | Q: {query[:60]}")

        results.append({
            "model":         model_name,
            "query":         query,
            "prediction":    pred,
            "answer":        gt_answer,
            "EM":            em,
            "F1":            f1,
            "Recall@K":      recall,
            "Hallucination": hallucination,
            "Confidence":    0.5 * float(np.max(scores)) + 0.5 * (1 - hallucination),
            "BLEU": bleu,
            "ROUGE-1": rouge1,
            "ROUGE-L": rougeL,
            "BERT-P": bert_p,
            "BERT-R": bert_r,
            "BERT-F1": bert_f1,
        })

    del model, tok
    torch.cuda.empty_cache()
    gc.collect()

# ============================
# 11. SAVE & SUMMARISE
# ============================
df = pd.DataFrame(results)
df.to_csv("rag_results_techqa.csv", index=False)

summary = df.groupby("model").agg({
    "EM": "mean",
    "F1": "mean",
    "Recall@K": "mean",
    "Hallucination": "mean",
    "Confidence": "mean",
    "BLEU": "mean",
    "ROUGE-1": "mean",
    "ROUGE-L": "mean",
    "BERT-F1": "mean",
})

print("\nSUMMARY:\n", summary)
summary.to_csv("summary_techqa.csv")

Generating train split:   0%|          | 0/910 [00:00<?, ? examples/s]

Total valid docs: 496


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

[140] EM:0 | F1:0.75 | BERT:0.94 | Hal:1.000 | Q: A .NET API error is thrown when attempting to install ICC 4.


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[141] EM:0 | F1:0.18 | BERT:0.86 | Hal:1.000 | Q: How do I downgrade an IBM Gateway, DataPower, appliance to a


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[144] EM:0 | F1:0.30 | BERT:0.82 | Hal:0.042 | Q: Help with Security Bulletin: The WebAdmin context for WMB V8
[145] EM:0 | F1:0.22 | BERT:0.84 | Hal:1.000 | Q: How can I obtain a Java thread dump against an execution gro

SUMMARY:
                                                  EM        F1  Recall@K  \
model                                                                     
unsloth/Qwen2.5-7B-Instruct-bnb-4bit       0.025862  0.254825  0.887931   
unsloth/mistral-7b-instruct-v0.3-bnb-4bit  0.025862  0.313289  0.887931   

                                           Hallucination  Confidence  \
model                                                                  
unsloth/Qwen2.5-7B-Instruct-bnb-4bit            0.938526    0.356471   
unsloth/mistral-7b-instruct-v0.3-bnb-4bit       0.883116    0.384177   

                                               BLEU   ROUGE-1   ROUGE-L  \
model                                                                     
unsloth/Qwen2.5-7B-Instruct